In [ ]:
!pip install -q streamlit datasets sqlalchemy langchain langchain-community langchain-groq openpyxl "requests==2.32.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.12 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.1 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which is incompatible.
langgraph 1.2.19 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from datasets import load_dataset
from sqlalchemy import create_engine, Table, MetaData, Column, String, DateTime, Integer, Float, ForeignKey

# =====================================================================
# 0. CLONAR OS REPOSITÓRIOS DO GITHUB AUTOMATICAMENTE NO COLAB
# =====================================================================
print("Baixando Dataset1 e Dataset2 diretamente do seu GitHub...")
!rm -rf /content/Dataset1 /content/Dataset2
!git clone -q https://github.com/scripttutoria/Dataset1.git /content/Dataset1
!git clone -q https://github.com/scripttutoria/Dataset2.git /content/Dataset2

def encontrar_excel(pasta):
    arquivos = glob.glob(f"{pasta}/**/*.xlsx", recursive=True) + glob.glob(f"{pasta}/**/*.xls", recursive=True)
    return arquivos[0] if arquivos else None

caminho_ds1 = encontrar_excel("/content/Dataset1")
caminho_ds2 = encontrar_excel("/content/Dataset2")
print(f"📂 Arquivo encontrado em Dataset1 (Fast Food): {caminho_ds1}")
print(f"📂 Arquivo encontrado em Dataset2 (Delivery 50k): {caminho_ds2}")

def criar_banco_sql(df_products, df_reviews, db_name):
    engine = create_engine(f"sqlite:///{db_name}")
    metadata = MetaData()

    products_table = Table(
        "products", metadata,
        Column("id_product", String, primary_key=True),
        Column("name", String, nullable=False),
        Column("category_or_menu", String),
        Column("average_rating", Float)
    )

    reviews_table = Table(
        "reviews", metadata,
        Column("id", Integer, primary_key=True, autoincrement=True),
        Column("rating", Float),
        Column("title_review", String),
        Column("text", String),
        Column("user_id", String),
        Column("date", DateTime),
        Column("verified_purchase", Integer),
        Column("store", String),
        Column("id_product", String, ForeignKey("products.id_product"))
    )

    reviews_table.drop(engine, checkfirst=True)
    products_table.drop(engine, checkfirst=True)
    metadata.create_all(engine)

    df_products.to_sql("products", con=engine, if_exists="append", index=False)
    df_reviews.to_sql("reviews", con=engine, if_exists="append", index=False)
    print(f"✅ Banco '{db_name}' criado com {len(df_products)} categorias/itens e {len(df_reviews)} avaliações!")

# =====================================================================
# 1. PROCESSAR DATASET 2 (Delivery 50k -> delivery_50k.db)
# =====================================================================
if caminho_ds2:
    print("\n1. Processando Dataset2 (Operação de Delivery 50k)...")
    df_del = pd.read_excel(caminho_ds2).drop_duplicates(subset=['reseña_bruta'])
    # Amostra de 10.000 registros únicos para consultas SQL instantâneas
    df_del = df_del.sample(n=min(10000, len(df_del)), random_state=42).reset_index(drop=True)

    nomes_temas = {
        'tiempo_entrega': 'Logística: Tempo de Entrega (Atrasos vs Rapidez)',
        'calidad_comida': 'Restaurante: Qualidade e Temperatura da Comida',
        'errores_pedido': 'Operação: Erros de Pedido e Itens Faltantes',
        'soporte_cliente': 'Atendimento: Suporte ao Cliente e Reembolsos',
        'usabilidad_app': 'Tecnologia: Usabilidade e Performance do App',
        'precio': 'Financeiro: Preços, Taxas de Entrega e Cupons',
        'cobertura_zona': 'Expansão: Cobertura de Entrega por Região'
    }
    df_del['nome_item'] = df_del['tema_principal'].map(nomes_temas).fillna(df_del['tema_principal'])

    df_prod_del = df_del.groupby(['tema_principal', 'nome_item'], as_index=False).agg(
        average_rating=('rating', lambda x: round(x.mean(), 2))
    ).rename(columns={'tema_principal': 'id_product', 'nome_item': 'name'})
    df_prod_del['category_or_menu'] = "Pilar Operacional de Food Delivery"
    df_prod_del = df_prod_del[['id_product', 'name', 'category_or_menu', 'average_rating']]

    np.random.seed(42)
    datas_del = pd.to_datetime(np.random.choice(pd.date_range("2025-01-01", "2026-03-01"), size=len(df_del)))

    df_rev_del = pd.DataFrame({
        'rating': df_del['rating'].astype(float),
        'title_review': df_del['tema_principal'].str.replace('_', ' ').str.title(),
        'text': df_del['reseña_bruta'],
        'user_id': [f"CLIENTE_{10000 + i}" for i in range(len(df_del))],
        'date': datas_del,
        'verified_purchase': 1,
        'store': "App Delivery Latam",
        'id_product': df_del['tema_principal']
    })
    criar_banco_sql(df_prod_del, df_rev_del, "/content/delivery_50k.db")

# =====================================================================
# 2. PROCESSAR DATASET 1 (Fast Food Centers -> fastfood_reviews.db)
# =====================================================================
if caminho_ds1:
    print("\n2. Processando Dataset1 (Fast Food Centers)...")
    df_ff = pd.read_excel(caminho_ds1)

    # Higienização das notas para garantir consistência entre texto e estrelas
    def corrigir_nota(row):
        txt = str(row['Review']).lower()
        if any(w in txt for w in ['loved', 'great', 'delicious', 'amazing', 'perfect', 'friendly']):
            return float(np.random.choice([4.0, 5.0]))
        elif any(w in txt for w in ['not happy', 'slow', 'wrong', 'cold', 'greasy']):
            return float(np.random.choice([1.0, 2.0, 3.0]))
        return float(row['Rating'])

    np.random.seed(42)
    df_ff['Rating_Clean'] = df_ff.apply(corrigir_nota, axis=1)
    df_ff['id_prod'] = "ITEM_" + df_ff['Favorite Item'].str.upper().str.replace(' ', '_')

    df_prod_ff = df_ff.groupby(['id_prod', 'Favorite Item'], as_index=False).agg(
        average_rating=('Rating_Clean', lambda x: round(x.mean(), 2))
    ).rename(columns={'id_prod': 'id_product', 'Favorite Item': 'name'})
    df_prod_ff['category_or_menu'] = "Cardápio Fast Food / Lanches"
    df_prod_ff = df_prod_ff[['id_product', 'name', 'category_or_menu', 'average_rating']]

    df_rev_ff = pd.DataFrame({
        'rating': df_ff['Rating_Clean'],
        'title_review': "Pedido de " + df_ff['Favorite Item'] + " (" + df_ff['Gender'] + ")",
        'text': df_ff['Review'],
        'user_id': "CUST_" + df_ff['Customer id'].astype(str),
        'date': pd.to_datetime(df_ff['Date of Visit'], errors='coerce'),
        'verified_purchase': 1,
        'store': "Rede Fast Food Parceira",
        'id_product': df_ff['id_prod']
    })
    criar_banco_sql(df_prod_ff, df_rev_ff, "/content/fastfood_reviews.db")

# =====================================================================
# 3. BASE RESTAURANTES & CARDÁPIOS (deelow/restaurant-reviews)
# =====================================================================
print("\n3. Processando deelow/restaurant-reviews...")
ds_rest = load_dataset("deelow/restaurant-reviews", split="train")
df_rest = ds_rest.to_pandas()
df_rest['stars'] = pd.to_numeric(df_rest['stars'], errors='coerce').fillna(4.0)
df_rest['average_review'] = pd.to_numeric(df_rest['average_review'], errors='coerce').fillna(4.0)

unique_rests = df_rest[['name', 'menu', 'average_review']].drop_duplicates(subset=['name']).reset_index(drop=True)
unique_rests['id_product'] = [f"REST_{100 + i}" for i in range(len(unique_rests))]

df_products_rest = unique_rests.rename(columns={
    'menu': 'category_or_menu',
    'average_review': 'average_rating'
})[['id_product', 'name', 'category_or_menu', 'average_rating']]

map_rest = dict(zip(df_products_rest['name'], df_products_rest['id_product']))
datas_simuladas = pd.to_datetime(np.random.choice(pd.date_range("2025-01-01", "2026-03-01"), size=len(df_rest)))

df_reviews_rest = pd.DataFrame({
    'rating': df_rest['stars'],
    'title_review': df_rest['review'].str.slice(0, 35) + "...",
    'text': df_rest['review'],
    'user_id': [f"USER_{1000 + i}" for i in range(len(df_rest))],
    'date': datas_simuladas,
    'verified_purchase': 1,
    'store': df_rest['name'],
    'id_product': df_rest['name'].map(map_rest)
})
criar_banco_sql(df_products_rest, df_reviews_rest, "/content/restaurant_reviews.db")

Baixando Dataset1 e Dataset2 diretamente do seu GitHub...
📂 Arquivo encontrado em Dataset1 (Fast Food): /content/Dataset1/Dataset1.xlsx
📂 Arquivo encontrado em Dataset2 (Delivery 50k): /content/Dataset2/Dataset2.xlsx

1. Processando Dataset2 (Operação de Delivery 50k)...
✅ Banco '/content/delivery_50k.db' criado com 7 categorias/itens e 10000 avaliações!

2. Processando Dataset1 (Fast Food Centers)...
✅ Banco '/content/fastfood_reviews.db' criado com 10 categorias/itens e 700 avaliações!

3. Processando deelow/restaurant-reviews...


README.md:   0%|          | 0.00/885 [00:00<?, ?B/s]

reviews.json:   0%|          | 0.00/4.29k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13 [00:00<?, ? examples/s]

✅ Banco '/content/restaurant_reviews.db' criado com 13 categorias/itens e 13 avaliações!


In [ ]:
%%writefile app.py
import os
import time
import streamlit as st
import pandas as pd
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_groq import ChatGroq

st.set_page_config(page_title="iFood Analytics Copilot | SQL Agent", page_icon="🍔", layout="wide")

with st.sidebar:
    st.title("🍔 iFood Data Copilot")
    st.caption("Agente Text-to-SQL para Operações de Delivery, Cardápios e SAC")
    st.divider()

    opcoes_bancos = {
        "🛵 Operação de Delivery (50k Avaliações - Temas/Logística)": (
            "/content/delivery_50k.db",
            "Operação de App de Delivery (focado em tempo de entrega, qualidade da comida, erros de pedido, suporte, app, preço e cobertura)"
        ),
        "🍟 Redes de Fast Food (Itens de Cardápio & Clientes)": (
            "/content/fastfood_reviews.db",
            "Redes de Fast Food Parceiras (focado em avaliação por item do cardápio como Burger, Pizza, Chicken Wings, Fries)"
        ),
        "🍽️ Restaurantes & Cardápios (deelow/restaurant-reviews)": (
            "/content/restaurant_reviews.db",
            "Restaurantes e Cardápios Completos (focado em pratos do menu e notas das lojas)"
        )
    }

    base_selecionada = st.selectbox("📊 Selecione o Contexto de Dados:", options=list(opcoes_bancos.keys()))
    db_path, contexto_negocio = opcoes_bancos[base_selecionada]
    nome_arquivo_db = os.path.basename(db_path)

    st.divider()
    api_key = st.text_input("🔑 Insira sua API Key da Groq (gsk_...):", type="password").strip()

    modelos_recomendados = [
        "openai/gpt-oss-120b",
        "llama-3.3-70b-versatile",
        "openai/gpt-oss-20b",
        "llama-3.1-8b-instant",
        "gemma2-9b-it"
    ]
    modelo_escolhido = st.selectbox("🤖 Modelo de IA:", options=modelos_recomendados, index=0)

    if st.button("🗑️ Limpar Histórico do Chat", use_container_width=True):
        st.session_state.messages = [
            {"role": "assistant", "content": f"🧹 Histórico limpo! Estou conectado ao banco **`{nome_arquivo_db}`** ({base_selecionada}). Qual análise deseja fazer?"}
        ]
        st.rerun()

    st.divider()
    st.markdown("### 💡 Perguntas Sugeridas para Teste")
    if "delivery_50k" in db_path:
        st.markdown("""
        - *Qual pilar operacional (`name`) tem a pior média de avaliação e quantas notas 1 estrela cada um recebeu?*
        - *Traga 5 exemplos de reclamações reais sobre `tiempo_entrega` e `errores_pedido` e sugira um plano de ação para o iFood.*
        """)
    elif "fastfood" in db_path:
        st.markdown("""
        - *Qual é o ranking dos itens do cardápio (`name`) por quantidade de pedidos e nota média?*
        - *Quais são as principais críticas nas avaliações de nota baixa e quais pratos estão envolvidos?*
        """)
    else:
        st.markdown("""
        - *Quais restaurantes têm pratos com 'chicken' no cardápio (`category_or_menu`)?*
        - *Qual restaurante tem a pior nota e qual foi a reclamação?*
        """)

@st.cache_resource
def carregar_conexao_sql(caminho_db):
    engine = create_engine(f"sqlite:///{caminho_db}")
    db = SQLDatabase(engine=engine, sample_rows_in_table_info=2)
    return engine, db

if not os.path.exists(db_path):
    st.warning(f"⚠️ O banco `{nome_arquivo_db}` ainda não foi gerado. Rode a Célula 2 no Colab!")
    st.stop()

engine, db = carregar_conexao_sql(db_path)

st.header(f"Painel Analítico: {base_selecionada}")

df_prod_view = pd.read_sql("SELECT * FROM products", engine)
df_rev_view = pd.read_sql("SELECT * FROM reviews", engine)

col1, col2, col3, col4 = st.columns(4)
col1.metric("Categorias / Lojas / Pratos", len(df_prod_view))
col2.metric("Total de Avaliações no SQL", len(df_rev_view))
col3.metric("Nota Média Geral", f"{df_rev_view['rating'].mean():.2f} ⭐")
col4.metric("Avaliações Críticas (1 e 2 ⭐)", len(df_rev_view[df_rev_view['rating'] <= 2]))

with st.expander(f"🔍 Inspecionar Tabelas Relacionais SQL (`{nome_arquivo_db}` -> `products` & `reviews`) e Distribuição", expanded=True):
    tab1, tab2, tab3 = st.tabs(["📦 Tabela: products", "💬 Tabela: reviews (Amostra)", "📈 Distribuição de Notas"])
    with tab1:
        st.dataframe(df_prod_view, use_container_width=True)
    with tab2:
        st.dataframe(df_rev_view.head(100), use_container_width=True)
    with tab3:
        contagem_notas = df_rev_view['rating'].astype(int).value_counts().sort_index()
        df_grafico = pd.DataFrame({
            "Nota (Estrelas)": [f"{nota} ⭐" for nota in contagem_notas.index],
            "Quantidade de Avaliações": contagem_notas.values
        }).set_index("Nota (Estrelas)")
        st.bar_chart(df_grafico)

st.divider()
st.subheader("💬 Converse com o Agente SQL de Food Delivery")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": f"Olá! Estou conectado ao banco **`{nome_arquivo_db}`** (*{base_selecionada}*). Qual análise operacional deseja fazer?"}
    ]
    st.session_state.banco_atual = nome_arquivo_db
elif st.session_state.get("banco_atual") != nome_arquivo_db:
    st.session_state.banco_atual = nome_arquivo_db
    st.session_state.messages.append({
        "role": "assistant",
        "content": f"🔄 **Base de dados alterada!** Agora estou conectado ao banco **`{nome_arquivo_db}`** (*{base_selecionada}*) com **{len(df_rev_view)} avaliações** disponíveis. O que deseja consultar nesta base?"
    })

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

pergunta_usuario = st.chat_input("Faça uma pergunta estratégica sobre entregas, cardápios ou reclamações...")

if pergunta_usuario:
    st.session_state.messages.append({"role": "user", "content": pergunta_usuario})
    with st.chat_message("user"):
        st.markdown(pergunta_usuario)

    if not api_key:
        st.warning("⚠️ Insira sua API Key da Groq na barra lateral esquerda.")
        st.stop()

    with st.chat_message("assistant"):
        msg_loading = f"Executando queries SQL em `{nome_arquivo_db}` com `{modelo_escolhido}`..."
        with st.spinner(msg_loading):
            llm = ChatGroq(model=modelo_escolhido, api_key=api_key, temperature=0, max_retries=3)

            prefixo_prompt = f"""
            Você é um Staff Data Scientist e Especialista em Inteligência de Negócios do iFood, analisando o banco `{nome_arquivo_db}` focado em: {contexto_negocio}.
            Tabelas disponíveis no SQLite:
            1. `products`: `id_product` (PK), `name` (nome do pilar operacional, prato ou restaurante), `category_or_menu`, `average_rating`.
            2. `reviews`: `id`, `rating` (1 a 5), `title_review`, `text` (texto da avaliação do cliente, pode estar em espanhol ou inglês), `user_id`, `date`, `verified_purchase`, `store`, `id_product` (FK).

            REGRAS OBRIGATÓRIAS PARA EVITAR ESTOURO DE TOKENS (ERROR 429) E FORMATAR BEM:
            - Sempre faça JOIN usando `id_product` quando precisar ligar `products` e `reviews`.
            - NUNCA faça SELECT de textos (`text` ou `title_review`) sem colocar `LIMIT 15` ou `LIMIT 20` na query SQL! Para contagens e médias use `GROUP BY` e `COUNT(*)` diretamente no SQL.
            - Mesmo que os textos originais estejam em espanhol ou inglês, traduza os insights e responda SEMPRE 100% em Português do Brasil.
            - Formate qualquer tabela usando Markdown padrão bem espaçado (pule uma linha em branco antes e depois da tabela).
            - Termine sempre com recomendações acionáveis para a operação de Food Delivery.
            """

            try:
                agente_sql = create_sql_agent(
                    llm=llm,
                    db=db,
                    agent_type="tool-calling",
                    prefix=prefixo_prompt,
                    verbose=True
                )
                resposta = agente_sql.invoke({"input": pergunta_usuario})
                saida = resposta["output"]
            except Exception as e:
                erro_str = str(e)
                if "429" in erro_str or "rate_limit" in erro_str.lower():
                    st.info("⏳ Limite de tokens por minuto (8.000 TPM) atingido na Groq. Aguardando 15 segundos para tentar novamente de forma automática...")
                    time.sleep(15)
                    try:
                        resposta = agente_sql.invoke({"input": pergunta_usuario})
                        saida = resposta["output"]
                    except Exception as e2:
                        saida = f"⚠️ O modelo `{modelo_escolhido}` possui um limite gratuito curto de 8.000 tokens/minuto na Groq. Dica: mude o seletor na barra lateral para **`llama-3.3-70b-versatile`** ou **`openai/gpt-oss-20b`** ou aguarde 20 segundos e envie novamente.\n\nDetalhe técnico: `{str(e2)}`"
                else:
                    saida = f"❌ Erro ao processar com `{modelo_escolhido}`: {erro_str}"

            st.markdown(saida)
            st.session_state.messages.append({"role": "assistant", "content": saida})

Overwriting app.py


In [ ]:
import time
import re

# Encerra processos antigos
!pkill -f streamlit
!pkill -f cloudflared

# Baixa o Cloudflared caso ainda não exista
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Inicia o Streamlit e o túnel em segundo plano (background)
!nohup streamlit run app.py --server.port 8501 > /content/streamlit_logs.txt 2>&1 &
!nohup ./cloudflared tunnel --url http://localhost:8501 > /content/cloudflare_logs.txt 2>&1 &

print("⏳ Iniciando servidor e gerando link público (aguarde 8 segundos)...")
time.sleep(8)

# Lê o log e exibe a URL do túnel sem travar a célula
with open("/content/cloudflare_logs.txt", "r") as f:
    logs = f.read()
    urls = re.findall(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if urls:
        print(f"✅ Acesse seu painel Streamlit aqui: {urls[0]}")
    else:
        print("⚠️ O link ainda está sendo gerado. Rode apenas: !grep -o 'https://.*trycloudflare.com' /content/cloudflare_logs.txt")

⏳ Iniciando servidor e gerando link público (aguarde 8 segundos)...
✅ Acesse seu painel Streamlit aqui: https://ira-acceptance-racial-uncertainty.trycloudflare.com
